In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
import numpy as np 
from scipy import optimize
from scipy.optimize import fsolve
import numpy as np
from vmbpy import *
from matplotlib import pyplot as plt



In [ ]:
    
def gaussian(x, amplitude, mean, waist, background):
    """1D Gaussian function with a background offset."""
    return amplitude * np.exp(-2 * ((x - mean) / waist)**2) + background

def f(x,waist1,waist2,distance, lambda_square_in_meter_div_pi_square):
    f1=x[0]*np.sqrt(1+lambda_square_in_meter_div_pi_square/(x[0]**4)*(x[1]**2))-waist1
    f2=x[0]*np.sqrt(1+lambda_square_in_meter_div_pi_square/(x[0]**4)*((x[1]+distance)**2))-waist2
    return [f1,f2]

def get_frames(n):
    # Open VmbPy system
    with VmbSystem.get_instance() as vmb:
        cams = vmb.get_all_cameras()
        if not cams:
            print("No cameras found.")
            return None

        # Use first detected camera
        with cams[0] as cam:
            print("Using camera:", cam.get_name())

            captured_frames = []

            # Loop n times to capture n frames
            for i in range(n):
                # Acquire single frame
                frame = cam.get_frame()

                # Convert and RE-ASSIGN
                # This creates a new 8-bit frame from the 14-bit original
                frame_mono8 = frame.convert_pixel_format(PixelFormat.Mono8)

                # Convert to OpenCV/Numpy format
                img = frame_mono8.as_opencv_image()
                
                # Append the numpy array to our list
                captured_frames.append(img)
            
            # Convert the list of frames into a single N-dimensional numpy array
            # The resulting shape will typically be (n, height, width, channels)
            return np.array(captured_frames)


In [ ]:
# get frames at location 1
frames_location_1 = get_frames(10)

In [ ]:
# get frames at location 2
frames_location_2 = get_frames(10)

In [ ]:
filename_1 = 'data\Data_profile_xy_#001_f_1.txt'
data = load_data(filename_1)

data_xf = data.iloc[:, [0, 1]].values  # camera frame 
data_yf = data.iloc[:, [2, 3]].values

filename_2 = 'data\Data_profile_xy_#001_n_1.txt'
data = load_data(filename_2)

data_xn = data.iloc[:, [0, 1]].values   
data_yn = data.iloc[:, [2, 3]].values

fit_xf, _ = optimize.curve_fit(gaussian, data_xf[:, 0], data_xf[:, 1])
fit_yf, _ = optimize.curve_fit(gaussian, data_yf[:, 0], data_yf[:, 1])
fit_xn, _ = optimize.curve_fit(gaussian, data_xn[:, 0], data_xn[:, 1])
fit_yn, _ = optimize.curve_fit(gaussian, data_yn[:, 0], data_yn[:, 1])

waist_far_x = fit_xf[2]*1e-6  # Convert to meters
waist_far_y = fit_yf[2]*1e-6
waist_near_x = fit_xn[2]*1e-6
waist_near_y = fit_yn[2]*1e-6

lambda_square_in_meter_div_pi_square=(1550e-9)**2/(np.pi)**2
distance = 0.010  # Distance between the two measurement points in meters
g_waist = 1e-5
g_location = 1e-2

calculated_waist_x = fsolve(f, [g_waist, g_location], args=(waist_near_x, waist_far_x, distance, lambda_square_in_meter_div_pi_square))
calculated_waist_y = fsolve(f, [g_waist, g_location], args=(waist_near_y, waist_far_y, distance, lambda_square_in_meter_div_pi_square))

print('calculated waist x:', calculated_waist_x[0])
print('calculated waist y:', calculated_waist_y[0])



